# DATA209 — Advanced Exploratory Data Analysis
# Practical P23-24 · Transformation and scaling

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 12 · Module 4 · CO4

---

**Objective.** Apply transformations and scalers, and compare the distribution before and after each one instead of assuming it worked.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 and P15-16 — the cleaned dataset and the modelling column list.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P15-16
df_clean = df.drop_duplicates().reset_index(drop=True)
for c in ["VisitorType", "Month"]:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

# --- the numeric modelling columns
model_cols = ["Administrative", "Administrative_Duration", "Informational",
              "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
              "BounceRates", "ExitRates", "PageValues"]

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P23-24 — Transformation and scaling

### Apply transformations

**Transformation** changes the *shape* of a distribution (non-linear: log, sqrt, Box-Cox).
**Scaling** changes its *location and spread* (linear: subtract a centre, divide by a spread).
Skew survives scaling untouched — so transform first if shape is the problem.

In [ ]:
# ---- Which columns need it? --------------------------------------------
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                   RobustScaler, PowerTransformer, QuantileTransformer)

model_cols = ["Administrative", "Administrative_Duration", "Informational",
              "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
              "BounceRates", "ExitRates", "PageValues"]

skew_before = df_clean[model_cols].skew().sort_values(ascending=False)
print("Skew before transformation")
print(skew_before.round(3).to_string())
needs = skew_before[skew_before.abs() > 1].index.tolist()
print(f"\n{len(needs)} columns exceed |skew| = 1: {needs}")

In [ ]:
# ---- Compare transforms on one column, then measure --------------------
col = "ProductRelated_Duration"
s   = df_clean[col]

yj = PowerTransformer(method="yeo-johnson")            # handles zeros and negatives
qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                         random_state=RANDOM_STATE)

variants = {
    "original"      : s,
    "log1p"         : np.log1p(s),
    "sqrt"          : np.sqrt(s),
    "yeo-johnson"   : pd.Series(yj.fit_transform(s.to_frame()).ravel(), index=s.index),
    "quantile-normal": pd.Series(qt.fit_transform(s.to_frame()).ravel(), index=s.index),
}

table = pd.DataFrame({
    "skew"     : {k: v.skew() for k, v in variants.items()},
    "kurtosis" : {k: v.kurtosis() for k, v in variants.items()},
    "mean"     : {k: v.mean() for k, v in variants.items()},
    "std"      : {k: v.std() for k, v in variants.items()},
})
table["|skew| reduction %"] = (
    (abs(table.loc["original", "skew"]) - table["skew"].abs())
    / abs(table.loc["original", "skew"]) * 100)
print(table.round(3).to_string())

fig, axes = plt.subplots(1, 5, figsize=(15, 2.9))
for ax, (name, v) in zip(axes, variants.items()):
    sns.histplot(v, bins=50, ax=ax, color="#6B4C7A")
    ax.set_title(f"{name}\nskew {v.skew():.2f}", fontsize=9); ax.set_xlabel("")
plt.tight_layout(); plt.show()

print("\nApply, then RE-MEASURE. A transform that does not reduce skew has not helped.")

In [ ]:
# ---- Transform every skewed column and re-check -------------------------
df_t = df_clean.copy()
for c in needs:
    df_t[c] = np.log1p(df_t[c].clip(lower=0))

skew_after = df_t[model_cols].skew()
compare = pd.DataFrame({"before": skew_before, "after": skew_after.reindex(skew_before.index)})
compare["improved"] = compare["after"].abs() < compare["before"].abs()
print(compare.round(3).to_string())

plt.figure(figsize=(9, 3.4))
idx = np.arange(len(compare)); w = 0.38
plt.bar(idx - w/2, compare["before"], w, label="before", color="#8B9199")
plt.bar(idx + w/2, compare["after"],  w, label="after log1p", color="#6B4C7A")
plt.xticks(idx, compare.index, rotation=35, ha="right"); plt.axhline(0, color="k", lw=.8)
plt.ylabel("skew"); plt.legend(); plt.title("Skew before and after transformation")
plt.tight_layout(); plt.show()

### Apply scaling and compare before/after

In [ ]:
# ---- Four scalers on the same data --------------------------------------
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler"  : MinMaxScaler(),
    "RobustScaler"  : RobustScaler(),
}

rows = []
for name, sc in scalers.items():
    out = pd.DataFrame(sc.fit_transform(df_t[model_cols]), columns=model_cols)
    rows.append({"scaler": name, "mean": out.values.mean(), "std": out.values.std(),
                 "min": out.values.min(), "max": out.values.max(),
                 "skew(PRD)": out["ProductRelated_Duration"].skew()})
rows.insert(0, {"scaler": "none", "mean": df_t[model_cols].values.mean(),
                "std": df_t[model_cols].values.std(), "min": df_t[model_cols].values.min(),
                "max": df_t[model_cols].values.max(),
                "skew(PRD)": df_t["ProductRelated_Duration"].skew()})
print(pd.DataFrame(rows).set_index("scaler").round(3).to_string())

print("\nNote the last column: every scaler leaves the skew unchanged.")
print("Scaling moves and stretches a distribution; it never reshapes it.")

In [ ]:
# ---- Why MinMaxScaler is fragile ---------------------------------------
demo = df_t[["ProductRelated_Duration"]].copy()
demo_out = demo.copy()
demo_out.iloc[0, 0] = demo.iloc[:, 0].max() * 25      # one extreme value

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for ax, (title, frame) in zip(axes, [("clean", demo), ("one extreme value added", demo_out)]):
    for name, sc in [("MinMax", MinMaxScaler()), ("Robust", RobustScaler())]:
        v = sc.fit_transform(frame).ravel()
        sns.kdeplot(v, ax=ax, label=name, lw=1.6)
    ax.set_title(title); ax.legend(); ax.set_xlim(-2, 3)
plt.tight_layout(); plt.show()

for title, frame in [("clean", demo), ("with outlier", demo_out)]:
    mm = MinMaxScaler().fit_transform(frame).ravel()
    rb = RobustScaler().fit_transform(frame).ravel()
    print(f"{title:14} MinMax median {np.median(mm):.4f} | Robust median {np.median(rb):.4f}")

print("\nOne extreme value stretches the MinMax range so every ordinary value compresses")
print("toward zero. RobustScaler uses the median and IQR and is barely affected.")

### Deliverable — P23-24

A notebook with a **skew-before / skew-after table**, the scaler comparison, the MinMax fragility
demonstration, and a justified choice of transform and scaler for every column you changed.